# 03. PromptTemplate과 Message

PromptTemplate을 사용하면 프롬프트를 재사용 가능한 템플릿으로 관리할 수 있습니다.

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

## 1. PromptTemplate - 단일 문자열 템플릿

In [2]:

from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘")

# 변수 채우기
prompt = template.invoke({"topic": "RAG"}) #꼭 딕셔너리 형태로 넣기
print(prompt)


text='RAG에 대해 한 문장으로 설명해줘'


## 2. ChatPromptTemplate - 역할 기반 템플릿

시스템/사용자/AI 메시지를 조합하여 템플릿을 만듭니다.

In [3]:


from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다."),
    ("human", "{question}")
])

messages = chat_prompt.invoke({
    "role": "파이썬",
    "question": "데코레이터가 뭔가요?"
})

print(messages)


messages=[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='데코레이터가 뭔가요?', additional_kwargs={}, response_metadata={})]


## 3. LLM과 연결하여 호출

In [4]:


from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

messages = chat_prompt.invoke({
    "role": "머신러닝",
    "question": "딥러닝이 뭔가요?"
})

response = llm.invoke(messages)
print(response.content)


딥러닝(Deep Learning)은 인공신경망(Artificial Neural Networks)을 기반으로 한 머신러닝의 한 분야로, 데이터에서 패턴을 학습하고 예측하는 데 사용됩니다. 딥러닝은 여러 층의 신경망을 통해 복잡한 데이터 표현을 학습할 수 있는 능력이 뛰어나며, 특히 이미지 인식, 자연어 처리, 음성 인식 등 다양한 분야에서 혁신적인 성과를 보여주고 있습니다.

딥러닝의 주요 특징은 다음과 같습니다:

1. **다층 구조**: 딥러닝 모델은 여러 개의 층(레이어)으로 구성되어 있으며, 각 층은 입력 데이터를 변환하여 다음 층으로 전달합니다. 이러한 다층 구조 덕분에 복잡한 데이터의 특징을 효과적으로 추출할 수 있습니다.

2. **자동 특징 학습**: 전통적인 머신러닝에서는 특징을 수동으로 추출해야 하지만, 딥러닝은 원시 데이터에서 자동으로 특징을 학습할 수 있습니다. 이는 데이터 전처리의 부담을 줄여줍니다.

3. **대량의 데이터 처리**: 딥러닝 모델은 대량의 데이터에서 학습할 때 성능이 향상됩니다. 따라서 대규모 데이터셋을 활용할 수 있는 환경에서 특히 효과적입니다.

4. **강력한 성능**: 이미지, 텍스트, 음성 등 다양한 유형의 데이터에서 높은 정확도를 달성할 수 있어, 많은 산업 분야에서 활용되고 있습니다.

딥러닝은 TensorFlow, PyTorch와 같은 다양한 프레임워크를 통해 구현할 수 있으며, 연구와 산업에서 활발히 사용되고 있습니다.


## 4. 대화 히스토리 포함 - MessagesPlaceholder

이전 대화 내용을 프롬프트에 삽입할 때 사용합니다.

In [15]:


from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# 이전 대화 기록
history = [
    HumanMessage(content="내 이름은 홍길동이고 500살이야"),
    AIMessage(content="안녕하세요, 홍길동님!")
]

messages = chat_prompt.invoke({
    "history": history,
    "question": "내가 몇살이게?"
})

response = llm.invoke(messages)
print(response.content)

홍길동님은 500살이라고 하셨습니다! 정말 놀라운 나이네요. 어떤 특별한 이야기가 있으신가요?


In [10]:

# 현재 대화를 history에 추가
history.append(HumanMessage(content="내 이름이 뭐라고?"))
history.append(AIMessage(content=response.content))
history

[HumanMessage(content='내 이름은 홍길동이고 5살이야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='홍길동님은 5살이라고 하셨죠! 정말 귀엽고 활기찬 나이네요. 어떤 놀이를 좋아하나요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 5. 다중 변수 템플릿 예제

In [12]:

template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {language} 번역가입니다."),
    ("human", "다음 문장을 번역해줘: {sentence}")
])

messages = template.invoke({
    "language": "French",
    "sentence": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

Aujourd'hui, le temps est vraiment agréable.


# 문제 해결

Q3. 번역 템플릿을 만들어 {source_language}, {target_language}, {text} 세 변수를 사용하는 다국어 번역 시스템을 구현해보세요.

In [14]:
template = ChatPromptTemplate.from_messages([
    ("system", """당신은 {sorted_language}에서 {target_language}로 번역하는 번역가입니다.
      출력을 할 때 {sorted_language}로 번역된 문장과 {target_language}로 번역된 문장을 모두 보여주세요.
     
      예시:
      입력(한국어): "오늘 날씨가 정말 좋네요.
      출력(영어): The weather is really nice today. 
     """),
    ("human", "다음 문장을 번역해줘: {text}")
])

messages = template.invoke({
    "sorted_language": "한국어",
    "target_language": "독일어",
    "text": "최선의 방어는 공격이다."
})

response = llm.invoke(messages)
print(response.content)

입력(한국어): "최선의 방어는 공격이다."  
출력(독일어): "Die beste Verteidigung ist der Angriff."
